In [ ]:
#!pip install optuna tqdm catboost scikit-learn tensorflow pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


#Оптимизация Optuna

In [ ]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np
from tqdm import tqdm
import optuna  # Импортируем Optuna

from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# --- НАСТРОЙКА CUDA ---
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)

# --- КОНФИГУРАЦИЯ ---
M = 30
SEQ_LENGTH = 30
EMA_FAST = 33
EMA_SLOW = 100
FREQ_MS = 60000
MIN_CHUNK_LENGTH = 100
ZIP_FILE = 'dataset_rework.zip'
OUTPUT_DIR = 'dataset_flattened'

# --- ПАРАМЕТРЫ РИСК-МЕНЕДЖМЕНТА ---
INITIAL_CAPITAL = 10000.0
FEE = 0.001                # 0.1%
POSITION_SIZE = 0.20       # 20%
RETURN_THRESH = FEE * 1.5

# --- [ФУНКЦИИ ПОДГОТОВКИ ДАННЫХ ОСТАЮТСЯ ПРЕЖНИМИ] ---
def unpack_dataset(zip_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Распаковка"):
            if file_info.is_dir() or '__MACOSX' in file_info.filename: continue
            flat_name = file_info.filename.replace('/', '_').replace('\\', '_')
            dest_path = os.path.join(output_dir, flat_name)
            if not os.path.exists(dest_path):
                with zip_ref.open(file_info) as src, open(dest_path, 'wb') as tgt:
                    tgt.write(src.read())

def get_contiguous_chunks(df, freq_ms=60000):
    df['symbol'] = df['symbol'].astype(str)
    df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    df['diff'] = df.groupby('symbol')['timestamp'].diff()
    df['is_new_chunk'] = (df['diff'] != freq_ms) | (df['symbol'] != df['symbol'].shift(1))
    df['chunk_id'] = df.groupby('symbol')['is_new_chunk'].cumsum()
    df['chunk_full_id'] = df['symbol'] + "_" + df['chunk_id'].astype(str)
    return df

def filter_and_report_chunks(df, min_len=100):
    chunk_stats = df.groupby('chunk_full_id').size().reset_index(name='length')
    valid_ids = chunk_stats[chunk_stats['length'] >= min_len]['chunk_full_id'].tolist()
    return df[df['chunk_full_id'].isin(valid_ids)].copy()

def create_financial_features(df, m_window=30, ema_fast=33, ema_slow=100):
    df = df.copy()
    grouped = df.groupby('chunk_full_id')
    df['log_return'] = grouped['close'].transform(lambda x: np.log(x / x.shift(1)))
    df['hl_spread'] = (df['high'] - df['low']) / df['close']
    df['body'] = (df['close'] - df['open']) / df['close']
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['close']
    df[f'dist_from_sma_{m_window}'] = grouped['close'].transform(lambda x: (x / x.rolling(m_window).mean()) - 1)
    grouped_new = df.groupby('chunk_full_id')
    df[f'volatility_{m_window}'] = grouped_new['log_return'].transform(lambda x: x.rolling(m_window).std())
    df['log_vol_change'] = grouped_new['volume'].transform(lambda x: np.log((x + 1) / (x.shift(1) + 1)))
    df[f'vol_vs_sma_{m_window}'] = grouped_new['volume'].transform(lambda x: x / (x.rolling(m_window).mean() + 1))
    df['future_return'] = grouped_new['close'].shift(-1) / df['close'] - 1
    df['rd_value_scaled'] = grouped_new['rd_value'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
    df['rd_value_diff'] = df['rd_value_scaled'].transform(lambda x: x.diff())
    df['rd_ema_fast'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_fast, adjust=False).mean())
    df['rd_ema_slow'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_slow, adjust=False).mean())
    df['rd_ema_diff'] = df['rd_ema_fast'] - df['rd_ema_slow']
    df['rd_ema_cross_signal'] = np.where(df['rd_ema_fast'] > df['rd_ema_slow'], 1, -1)

    base_3d_features = ['log_return', 'hl_spread', 'body', 'upper_shadow', 'lower_shadow',
                        f'dist_from_sma_{m_window}', f'volatility_{m_window}', 'log_vol_change', f'vol_vs_sma_{m_window}']

    lag_targets = ['log_return', 'hl_spread', 'body']
    lag_feats = {}
    print("Генерация лагов...")
    for col in tqdm(lag_targets, desc="Лаги"):
        for i in range(1, m_window + 1):
            lag_feats[f"{col}_lag_{i}"] = grouped_new[col].shift(i)

    df = pd.concat([df, pd.DataFrame(lag_feats)], axis=1)
    all_features = base_3d_features + ['rd_value_scaled', 'rd_value_diff', 'rd_ema_fast', 'rd_ema_slow', 'rd_ema_diff', 'rd_ema_cross_signal'] + list(lag_feats.keys())
    df = df.dropna(subset=all_features + ['future_return']).reset_index(drop=True)
    return df, sorted(all_features), base_3d_features

# --- УНИФИЦИРОВАННЫЙ БЭКТЕСТЕР ---
def advanced_backtest_chunked(raw_preds, future_returns, chunk_ids, model_name, silent=False):
    signals = np.where(raw_preds > RETURN_THRESH, 1, np.where(raw_preds < -RETURN_THRESH, -1, 0))
    target_positions = signals * POSITION_SIZE
    df_eval = pd.DataFrame({'chunk_id': chunk_ids, 'pos': target_positions, 'ret': future_returns})

    all_net_returns = []
    for cid, group in df_eval.groupby('chunk_id'):
        pos = group['pos'].values
        ret = group['ret'].values
        gross_returns = pos * ret
        prev_pos = np.insert(pos[:-1], 0, 0)
        position_changes = np.abs(pos - prev_pos)
        fee_cost = position_changes * FEE
        fee_cost[-1] += np.abs(pos[-1]) * FEE
        all_net_returns.extend(gross_returns - fee_cost)

    all_net_returns = np.array(all_net_returns)
    cumulative_equity = INITIAL_CAPITAL * np.cumprod(1 + all_net_returns)
    cumulative_equity = np.maximum(cumulative_equity, 0)

    final_balance = cumulative_equity[-1]
    pnl_percent = ((final_balance / INITIAL_CAPITAL) - 1) * 100

    if not silent:
        print(f"[{model_name:<30}] Баланс: ${final_balance:>8.2f} ({pnl_percent:>7.2f}%)")
    return pnl_percent

# --- OPTUNA OPTIMIZATION ---
def objective(trial, X_train, y_train, X_val, y_val, chunk_ids_val):
    # Гиперпараметры для поиска
    param = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 5.0),
        "bootstrap_type": "Bayesian",
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "od_type": "Iter",
        "od_wait": 50,
        "task_type": "GPU",
        "verbose": False
    }

    model = CatBoostRegressor(**param)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

    preds = model.predict(X_val)
    # Оптимизируем напрямую PnL на валидации
    pnl = advanced_backtest_chunked(preds, y_val, chunk_ids_val, "Optuna Trial", silent=True)
    return pnl

# --- ОСНОВНОЙ ПАЙПЛАЙН ---
def main():
    if os.path.exists(ZIP_FILE): unpack_dataset(ZIP_FILE, OUTPUT_DIR)

    files = glob.glob(os.path.join(OUTPUT_DIR, '*.csv'))
    df_list = [pd.read_csv(f, sep=';' if ';' in open(f).readline() else ',').assign(symbol=os.path.basename(f)) for f in tqdm(files[:100], desc="Загрузка")]
    df_raw = pd.concat(df_list, ignore_index=True)
    df_filtered = filter_and_report_chunks(get_contiguous_chunks(df_raw), min_len=MIN_CHUNK_LENGTH)
    df_feat, all_features, _ = create_financial_features(df_filtered)

    # Разделение: Train / Val (для Optuna) / Test (финал)
    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)
    n = len(df_feat)
    train_df = df_feat.iloc[:int(n*0.7)]
    val_df = df_feat.iloc[int(n*0.7):int(n*0.85)]
    test_df = df_feat.iloc[int(n*0.85):]

    features = [f for f in all_features if 'rd_' in f or 'lag' in f or f in ['log_return', 'hl_spread', 'body']]

    print("\n🚀 ЗАПУСК OPTUNA ОПТИМИЗАЦИИ (30 попыток)...")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, train_df[features], train_df['future_return'],
                                           val_df[features], val_df['future_return'], val_df['chunk_full_id']), n_trials=30)

    print("\n✅ Лучшие параметры:", study.best_params)

    # Финальное обучение с лучшими параметрами
    print("\n--- ФИНАЛЬНЫЙ ТЕСТ НА НЕВИДИМЫХ ДАННЫХ ---")
    best_model = CatBoostRegressor(**study.best_params, iterations=1000, task_type="GPU", verbose=False)
    best_model.fit(train_df[features], train_df['future_return'])

    final_preds = best_model.predict(test_df[features])
    advanced_backtest_chunked(final_preds, test_df['future_return'], test_df['chunk_full_id'], "Optimized CatBoost")

    # Сравнение с Buy & Hold на том же участке
    advanced_backtest_chunked(np.ones_like(final_preds)*100, test_df['future_return'], test_df['chunk_full_id'], "Buy & Hold (Market)")

if __name__ == "__main__":
    main()

Загрузка: 100%|██████████| 100/100 [00:00<00:00, 592.62it/s]


Генерация лагов...


Лаги: 100%|██████████| 3/3 [00:00<00:00, 51.35it/s]
[I 2026-03-04 13:49:01,232] A new study created in memory with name: no-name-1d38a6a1-da1b-49df-9b91-a612992ac5bb



🚀 ЗАПУСК OPTUNA ОПТИМИЗАЦИИ (30 попыток)...


[I 2026-03-04 13:49:06,965] Trial 0 finished with value: -0.23053220179687361 and parameters: {'learning_rate': 0.033496929057667796, 'depth': 4, 'l2_leaf_reg': 8.070211199503369, 'random_strength': 4.33311541341172, 'bagging_temperature': 0.6130784138187486}. Best is trial 0 with value: -0.23053220179687361.
[I 2026-03-04 13:49:39,011] Trial 1 finished with value: 0.7851935403562216 and parameters: {'learning_rate': 0.03962233059428771, 'depth': 10, 'l2_leaf_reg': 8.35234280680314, 'random_strength': 2.8123928397189197, 'bagging_temperature': 0.4116975973349618}. Best is trial 1 with value: 0.7851935403562216.
[I 2026-03-04 13:51:09,952] Trial 2 finished with value: 0.0 and parameters: {'learning_rate': 0.001285629516188118, 'depth': 10, 'l2_leaf_reg': 3.3290207493153936, 'random_strength': 3.301759713602354, 'bagging_temperature': 0.30603977794907944}. Best is trial 1 with value: 0.7851935403562216.
[I 2026-03-04 13:52:00,477] Trial 3 finished with value: -0.3858064713862741 and para


✅ Лучшие параметры: {'learning_rate': 0.032579022462332734, 'depth': 8, 'l2_leaf_reg': 1.01700156827774, 'random_strength': 2.3712067543173116, 'bagging_temperature': 0.5048896529867589}

--- ФИНАЛЬНЫЙ ТЕСТ НА НЕВИДИМЫХ ДАННЫХ ---
[Optimized CatBoost            ] Баланс: $ 9517.11 (  -4.83%)
[Buy & Hold (Market)           ] Баланс: $10415.05 (   4.15%)


#Эксперимент 1

In [ ]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np
from tqdm import tqdm

from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# --- НАСТРОЙКА CUDA ---
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)

# --- КОНФИГУРАЦИЯ ---
M = 30
SEQ_LENGTH = 30
EMA_FAST = 33
EMA_SLOW = 100
FREQ_MS = 60000
MIN_CHUNK_LENGTH = 100
ZIP_FILE = 'dataset_rework.zip'
OUTPUT_DIR = 'dataset_flattened'

# --- ПАРАМЕТРЫ РИСК-МЕНЕДЖМЕНТА ---
INITIAL_CAPITAL = 10000.0
FEE = 0.00055                # Комиссия 0.1%
POSITION_SIZE = 0.1       # Риск 10% от текущего депо
# Порог входа:
RETURN_THRESH = FEE * 1.5

# Лучшие параметры из Optuna
BEST_CB_PARAMS = {
    'learning_rate': 0.032579022462332734,
    'depth': 8,
    'l2_leaf_reg': 1.01700156827774,
    'random_strength': 2.3712067543173116,
    'bagging_temperature': 0.5048896529867589,
    'iterations': 1000,
    'task_type': 'GPU',
    'verbose': False,
    'early_stopping_rounds': 50
}

# --- 1. Вспомогательные функции ---
def unpack_dataset(zip_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Распаковка"):
            if file_info.is_dir() or '__MACOSX' in file_info.filename: continue
            flat_name = file_info.filename.replace('/', '_').replace('\\', '_')
            dest_path = os.path.join(output_dir, flat_name)
            if not os.path.exists(dest_path):
                with zip_ref.open(file_info) as src, open(dest_path, 'wb') as tgt:
                    tgt.write(src.read())

def get_contiguous_chunks(df, freq_ms=60000):
    df['symbol'] = df['symbol'].astype(str)
    df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    df['diff'] = df.groupby('symbol')['timestamp'].diff()
    df['is_new_chunk'] = (df['diff'] != freq_ms) | (df['symbol'] != df['symbol'].shift(1))
    df['chunk_id'] = df.groupby('symbol')['is_new_chunk'].cumsum()
    df['chunk_full_id'] = df['symbol'] + "_" + df['chunk_id'].astype(str)
    return df

def filter_and_report_chunks(df, min_len=100):
    chunk_stats = df.groupby('chunk_full_id').size().reset_index(name='length')
    valid_ids = chunk_stats[chunk_stats['length'] >= min_len]['chunk_full_id'].tolist()
    return df[df['chunk_full_id'].isin(valid_ids)].copy()

def create_financial_features(df, m_window=30, ema_fast=33, ema_slow=100):
    df = df.copy()
    grouped = df.groupby('chunk_full_id')
    df['log_return'] = grouped['close'].transform(lambda x: np.log(x / x.shift(1)))
    df['hl_spread'] = (df['high'] - df['low']) / df['close']
    df['body'] = (df['close'] - df['open']) / df['close']
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['close']
    df[f'dist_from_sma_{m_window}'] = grouped['close'].transform(lambda x: (x / x.rolling(m_window).mean()) - 1)

    grouped_new = df.groupby('chunk_full_id')
    df[f'volatility_{m_window}'] = grouped_new['log_return'].transform(lambda x: x.rolling(m_window).std())
    df['log_vol_change'] = grouped_new['volume'].transform(lambda x: np.log((x + 1) / (x.shift(1) + 1)))
    df[f'vol_vs_sma_{m_window}'] = grouped_new['volume'].transform(lambda x: x / (x.rolling(m_window).mean() + 1))
    df['future_return'] = grouped_new['close'].shift(-1) / df['close'] - 1

    df['rd_value_scaled'] = grouped_new['rd_value'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
    df['rd_value_diff'] = df['rd_value_scaled'].transform(lambda x: x.diff())
    df['rd_ema_fast'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_fast, adjust=False).mean())
    df['rd_ema_slow'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_slow, adjust=False).mean())
    df['rd_ema_diff'] = df['rd_ema_fast'] - df['rd_ema_slow']
    df['rd_ema_cross_signal'] = np.where(df['rd_ema_fast'] > df['rd_ema_slow'], 1, -1)

    base_3d_features = ['log_return', 'hl_spread', 'body', 'upper_shadow', 'lower_shadow',
                        f'dist_from_sma_{m_window}', f'volatility_{m_window}', 'log_vol_change', f'vol_vs_sma_{m_window}']

    lag_targets = ['log_return', 'hl_spread', 'body']
    lag_feats = {}
    for col in tqdm(lag_targets, desc="Лаги"):
        for i in range(1, m_window + 1):
            lag_feats[f"{col}_lag_{i}"] = grouped_new[col].shift(i)

    df = pd.concat([df, pd.DataFrame(lag_feats)], axis=1)
    all_features = base_3d_features + ['rd_value_scaled', 'rd_value_diff', 'rd_ema_fast', 'rd_ema_slow', 'rd_ema_diff', 'rd_ema_cross_signal'] + list(lag_feats.keys())
    df = df.dropna(subset=all_features + ['future_return']).reset_index(drop=True)
    return df, sorted(all_features), base_3d_features

# --- 2. Честный Бэктестер ---
def advanced_backtest_chunked(raw_preds, future_returns, chunk_ids, model_name):
    signals = np.where(raw_preds > RETURN_THRESH, 1, np.where(raw_preds < -RETURN_THRESH, -1, 0))
    target_positions = signals * POSITION_SIZE
    df_eval = pd.DataFrame({'chunk_id': chunk_ids, 'pos': target_positions, 'ret': future_returns})

    all_net_returns = []
    total_trades = 0

    for cid, group in df_eval.groupby('chunk_id'):
        pos, ret = group['pos'].values, group['ret'].values
        gross_returns = pos * ret
        prev_pos = np.insert(pos[:-1], 0, 0)
        position_changes = np.abs(pos - prev_pos)
        fee_cost = position_changes * FEE
        fee_cost[-1] += np.abs(pos[-1]) * FEE # Закрытие в конце чанка

        all_net_returns.extend(gross_returns - fee_cost)
        total_trades += np.sum(position_changes > 0) + (1 if np.abs(pos[-1]) > 0 else 0)

    all_net_returns = np.array(all_net_returns)
    cumulative_equity = INITIAL_CAPITAL * np.cumprod(1 + all_net_returns)
    cumulative_equity = np.maximum(cumulative_equity, 0)

    final_balance = cumulative_equity[-1]
    pnl_usd = final_balance - INITIAL_CAPITAL
    pnl_pct = (pnl_usd / INITIAL_CAPITAL) * 100

    print(f"[{model_name:<30}] Баланс: ${final_balance:>8.2f} | Профит: ${pnl_usd:>8.2f} ({pnl_pct:>7.2f}%) | Сделок: {int(total_trades)}")
    return pnl_pct

# --- 3. Обучение Моделей ---
def run_catboost(X_train, y_train, X_test, y_test, chunk_ids_test, model_name, features):
    model = CatBoostRegressor(**BEST_CB_PARAMS)
    model.fit(X_train[features], y_train, eval_set=(X_test[features], y_test))
    preds = model.predict(X_test[features])
    return advanced_backtest_chunked(preds, y_test, chunk_ids_test, model_name)

def run_keras_reg(X_train, y_train, X_test, y_test, chunk_ids_test, model_name, is_lstm=False, feats=None):
    if not is_lstm:
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train[feats])
        X_test_s = scaler.transform(X_test[feats])
        model = Sequential([
            Input(shape=(X_train_s.shape[1],)),
            Dense(128, activation='relu'), BatchNormalization(), Dropout(0.2),
            Dense(64, activation='relu'), Dense(1, activation='linear')
        ])
        model.compile(optimizer=Adam(0.0005), loss='mse')
        model.fit(X_train_s, y_train, validation_data=(X_test_s, y_test), epochs=50, batch_size=1024, verbose=0)
        preds = model.predict(X_test_s, verbose=0).flatten()
    else:
        # 3D подготовка внутри для краткости
        X_t3, y_t3, _ = create_3d_sequences(X_train, feats)
        X_v3, y_v3, c_v3 = create_3d_sequences(X_test, feats)
        # Решейп для скалера
        s = StandardScaler()
        X_t3_s = s.fit_transform(X_t3.reshape(-1, X_t3.shape[2])).reshape(X_t3.shape)
        X_v3_s = s.transform(X_v3.reshape(-1, X_v3.shape[2])).reshape(X_v3.shape)

        model = Sequential([
            Input(shape=(SEQ_LENGTH, len(feats))),
            LSTM(32, return_sequences=False), Dropout(0.2),
            Dense(16, activation='relu'), Dense(1, activation='linear')
        ])
        model.compile(optimizer=Adam(0.0005), loss='mse')
        model.fit(X_t3_s, y_t3, validation_data=(X_v3_s, y_v3), epochs=30, batch_size=512, verbose=0)
        preds = model.predict(X_v3_s, verbose=0).flatten()
        return advanced_backtest_chunked(preds, y_v3, c_v3, model_name)

    return advanced_backtest_chunked(preds, y_test, chunk_ids_test, model_name)

def create_3d_sequences(df, feature_cols):
    X, y, chunks = [], [], []
    for cid, group in df.groupby('chunk_full_id'):
        f_vals, t_vals = group[feature_cols].values, group['future_return'].values
        for i in range(len(group) - SEQ_LENGTH + 1):
            X.append(f_vals[i : i + SEQ_LENGTH])
            y.append(t_vals[i + SEQ_LENGTH - 1])
            chunks.append(cid)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(chunks)

# --- ОСНОВНОЙ ПАЙПЛАЙН ---
def main():
    if os.path.exists(ZIP_FILE): unpack_dataset(ZIP_FILE, OUTPUT_DIR)
    files = glob.glob(os.path.join(OUTPUT_DIR, '*.csv'))
    df_raw = pd.concat([pd.read_csv(f, sep=';' if ';' in open(f).readline() else ',').assign(symbol=os.path.basename(f)) for f in tqdm(files[:150], desc="Загрузка")], ignore_index=True)

    df_feat, all_f, base_3d = create_financial_features(filter_and_report_chunks(get_contiguous_chunks(df_raw), 100))

    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)
    split_idx = int(len(df_feat) * 0.8)
    train_df, test_df = df_feat.iloc[:split_idx], df_feat.iloc[split_idx:]

    y_test, c_test = test_df['future_return'].values, test_df['chunk_full_id'].values

    print("\n" + "="*80)
    print(f"💰 ФИНАНСОВЫЙ ОТЧЕТ (CUDA) | Старт: $10,000 | Риск: 20% | Порог: {RETURN_THRESH*100:.2f}%")
    print("="*80)

    # 1. Buy & Hold
    advanced_backtest_chunked(np.ones_like(y_test)*100, y_test, c_test, "0. Buy & Hold (Market)")

    # 2. Optimized CatBoost (With Signal)
    run_catboost(train_df, train_df['future_return'], test_df, y_test, c_test, "1. Optimized CB (Signal)", all_f)

    # 3. CatBoost (No Signal)
    f_no_rd = [f for f in all_f if 'rd_' not in f]
    run_catboost(train_df, train_df['future_return'], test_df, y_test, c_test, "2. Optimized CB (No Signal)", f_no_rd)

    # 4. Tabular DNN
    run_keras_reg(train_df, train_df['future_return'], test_df, y_test, c_test, "3. Tabular DNN (No Signal)", False, f_no_rd)

    # 5. LSTM
    run_keras_reg(train_df, train_df['future_return'], test_df, y_test, c_test, "4. LSTM (No Signal)", True, base_3d)

if __name__ == "__main__":
    main()

Лаги: 100%|██████████| 3/3 [00:00<00:00, 21.70it/s]



💰 ФИНАНСОВЫЙ ОТЧЕТ (CUDA) | Старт: $10,000 | Риск: 20% | Порог: 0.08%
[0. Buy & Hold (Market)        ] Баланс: $ 9967.06 | Профит: $  -32.94 (  -0.33%) | Сделок: 78
[1. Optimized CB (Signal)      ] Баланс: $10314.46 | Профит: $  314.46 (   3.14%) | Сделок: 661
[2. Optimized CB (No Signal)   ] Баланс: $ 9927.63 | Профит: $  -72.37 (  -0.72%) | Сделок: 156
[3. Tabular DNN (No Signal)    ] Баланс: $ 5439.55 | Профит: $-4560.45 ( -45.60%) | Сделок: 6848
[4. LSTM (No Signal)           ] Баланс: $10024.74 | Профит: $   24.74 (   0.25%) | Сделок: 147


#Эксперимент 2

In [ ]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np
from tqdm import tqdm

from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# --- НАСТРОЙКА CUDA ---
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)

# --- КОНФИГУРАЦИЯ ---
M = 30
SEQ_LENGTH = 30
EMA_FAST = 33
EMA_SLOW = 100
FREQ_MS = 60000
MIN_CHUNK_LENGTH = 100
ZIP_FILE = 'dataset_rework.zip'
OUTPUT_DIR = 'dataset_flattened'

# --- ПАРАМЕТРЫ РИСК-МЕНЕДЖМЕНТА ---
INITIAL_CAPITAL = 10000.0
FEE = 0.001                # Комиссия 0.1%
POSITION_SIZE = 0.05       # Риск 20% от текущего депо
# Порог входа: предсказание должно быть выше 0.2%, чтобы покрыть вход/выход (0.1% + 0.1%)
RETURN_THRESH = FEE * 2.0

# Лучшие параметры из Optuna
BEST_CB_PARAMS = {
    'learning_rate': 0.032579022462332734,
    'depth': 8,
    'l2_leaf_reg': 1.01700156827774,
    'random_strength': 2.3712067543173116,
    'bagging_temperature': 0.5048896529867589,
    'iterations': 1000,
    'task_type': 'GPU',
    'verbose': False,
    'early_stopping_rounds': 50
}

# --- 1. Вспомогательные функции ---
def unpack_dataset(zip_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Распаковка"):
            if file_info.is_dir() or '__MACOSX' in file_info.filename: continue
            flat_name = file_info.filename.replace('/', '_').replace('\\', '_')
            dest_path = os.path.join(output_dir, flat_name)
            if not os.path.exists(dest_path):
                with zip_ref.open(file_info) as src, open(dest_path, 'wb') as tgt:
                    tgt.write(src.read())

def get_contiguous_chunks(df, freq_ms=60000):
    df['symbol'] = df['symbol'].astype(str)
    df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    df['diff'] = df.groupby('symbol')['timestamp'].diff()
    df['is_new_chunk'] = (df['diff'] != freq_ms) | (df['symbol'] != df['symbol'].shift(1))
    df['chunk_id'] = df.groupby('symbol')['is_new_chunk'].cumsum()
    df['chunk_full_id'] = df['symbol'] + "_" + df['chunk_id'].astype(str)
    return df

def filter_and_report_chunks(df, min_len=100):
    chunk_stats = df.groupby('chunk_full_id').size().reset_index(name='length')
    valid_ids = chunk_stats[chunk_stats['length'] >= min_len]['chunk_full_id'].tolist()
    return df[df['chunk_full_id'].isin(valid_ids)].copy()

def create_financial_features(df, m_window=30, ema_fast=33, ema_slow=100):
    df = df.copy()
    grouped = df.groupby('chunk_full_id')
    df['log_return'] = grouped['close'].transform(lambda x: np.log(x / x.shift(1)))
    df['hl_spread'] = (df['high'] - df['low']) / df['close']
    df['body'] = (df['close'] - df['open']) / df['close']
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['close']
    df[f'dist_from_sma_{m_window}'] = grouped['close'].transform(lambda x: (x / x.rolling(m_window).mean()) - 1)

    grouped_new = df.groupby('chunk_full_id')
    df[f'volatility_{m_window}'] = grouped_new['log_return'].transform(lambda x: x.rolling(m_window).std())
    df['log_vol_change'] = grouped_new['volume'].transform(lambda x: np.log((x + 1) / (x.shift(1) + 1)))
    df[f'vol_vs_sma_{m_window}'] = grouped_new['volume'].transform(lambda x: x / (x.rolling(m_window).mean() + 1))
    df['future_return'] = grouped_new['close'].shift(-1) / df['close'] - 1

    df['rd_value_scaled'] = grouped_new['rd_value'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
    df['rd_value_diff'] = df['rd_value_scaled'].transform(lambda x: x.diff())
    df['rd_ema_fast'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_fast, adjust=False).mean())
    df['rd_ema_slow'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_slow, adjust=False).mean())
    df['rd_ema_diff'] = df['rd_ema_fast'] - df['rd_ema_slow']
    df['rd_ema_cross_signal'] = np.where(df['rd_ema_fast'] > df['rd_ema_slow'], 1, -1)

    base_3d_features = ['log_return', 'hl_spread', 'body', 'upper_shadow', 'lower_shadow',
                        f'dist_from_sma_{m_window}', f'volatility_{m_window}', 'log_vol_change', f'vol_vs_sma_{m_window}']

    lag_targets = ['log_return', 'hl_spread', 'body']
    lag_feats = {}
    for col in tqdm(lag_targets, desc="Лаги"):
        for i in range(1, m_window + 1):
            lag_feats[f"{col}_lag_{i}"] = grouped_new[col].shift(i)

    df = pd.concat([df, pd.DataFrame(lag_feats)], axis=1)
    all_features = base_3d_features + ['rd_value_scaled', 'rd_value_diff', 'rd_ema_fast', 'rd_ema_slow', 'rd_ema_diff', 'rd_ema_cross_signal'] + list(lag_feats.keys())
    df = df.dropna(subset=all_features + ['future_return']).reset_index(drop=True)
    return df, sorted(all_features), base_3d_features

# --- 2. Честный Бэктестер ---
def advanced_backtest_chunked(raw_preds, future_returns, chunk_ids, model_name):
    signals = np.where(raw_preds > RETURN_THRESH, 1, np.where(raw_preds < -RETURN_THRESH, -1, 0))
    target_positions = signals * POSITION_SIZE
    df_eval = pd.DataFrame({'chunk_id': chunk_ids, 'pos': target_positions, 'ret': future_returns})

    all_net_returns = []
    total_trades = 0

    for cid, group in df_eval.groupby('chunk_id'):
        pos, ret = group['pos'].values, group['ret'].values
        gross_returns = pos * ret
        prev_pos = np.insert(pos[:-1], 0, 0)
        position_changes = np.abs(pos - prev_pos)
        fee_cost = position_changes * FEE
        fee_cost[-1] += np.abs(pos[-1]) * FEE # Закрытие в конце чанка

        all_net_returns.extend(gross_returns - fee_cost)
        total_trades += np.sum(position_changes > 0) + (1 if np.abs(pos[-1]) > 0 else 0)

    all_net_returns = np.array(all_net_returns)
    cumulative_equity = INITIAL_CAPITAL * np.cumprod(1 + all_net_returns)
    cumulative_equity = np.maximum(cumulative_equity, 0)

    final_balance = cumulative_equity[-1]
    pnl_usd = final_balance - INITIAL_CAPITAL
    pnl_pct = (pnl_usd / INITIAL_CAPITAL) * 100

    print(f"[{model_name:<30}] Баланс: ${final_balance:>8.2f} | Профит: ${pnl_usd:>8.2f} ({pnl_pct:>7.2f}%) | Сделок: {int(total_trades)}")
    return pnl_pct

# --- 3. Обучение Моделей ---
def run_catboost(X_train, y_train, X_test, y_test, chunk_ids_test, model_name, features):
    model = CatBoostRegressor(**BEST_CB_PARAMS)
    model.fit(X_train[features], y_train, eval_set=(X_test[features], y_test))
    preds = model.predict(X_test[features])
    return advanced_backtest_chunked(preds, y_test, chunk_ids_test, model_name)

def run_keras_reg(X_train, y_train, X_test, y_test, chunk_ids_test, model_name, is_lstm=False, feats=None):
    if not is_lstm:
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train[feats])
        X_test_s = scaler.transform(X_test[feats])
        model = Sequential([
            Input(shape=(X_train_s.shape[1],)),
            Dense(128, activation='relu'), BatchNormalization(), Dropout(0.2),
            Dense(64, activation='relu'), Dense(1, activation='linear')
        ])
        model.compile(optimizer=Adam(0.0005), loss='mse')
        model.fit(X_train_s, y_train, validation_data=(X_test_s, y_test), epochs=50, batch_size=1024, verbose=0)
        preds = model.predict(X_test_s, verbose=0).flatten()
    else:
        # 3D подготовка внутри для краткости
        X_t3, y_t3, _ = create_3d_sequences(X_train, feats)
        X_v3, y_v3, c_v3 = create_3d_sequences(X_test, feats)
        # Решейп для скалера
        s = StandardScaler()
        X_t3_s = s.fit_transform(X_t3.reshape(-1, X_t3.shape[2])).reshape(X_t3.shape)
        X_v3_s = s.transform(X_v3.reshape(-1, X_v3.shape[2])).reshape(X_v3.shape)

        model = Sequential([
            Input(shape=(SEQ_LENGTH, len(feats))),
            LSTM(32, return_sequences=False), Dropout(0.2),
            Dense(16, activation='relu'), Dense(1, activation='linear')
        ])
        model.compile(optimizer=Adam(0.0005), loss='mse')
        model.fit(X_t3_s, y_t3, validation_data=(X_v3_s, y_v3), epochs=30, batch_size=512, verbose=0)
        preds = model.predict(X_v3_s, verbose=0).flatten()
        return advanced_backtest_chunked(preds, y_v3, c_v3, model_name)

    return advanced_backtest_chunked(preds, y_test, chunk_ids_test, model_name)

def create_3d_sequences(df, feature_cols):
    X, y, chunks = [], [], []
    for cid, group in df.groupby('chunk_full_id'):
        f_vals, t_vals = group[feature_cols].values, group['future_return'].values
        for i in range(len(group) - SEQ_LENGTH + 1):
            X.append(f_vals[i : i + SEQ_LENGTH])
            y.append(t_vals[i + SEQ_LENGTH - 1])
            chunks.append(cid)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(chunks)

# --- ОСНОВНОЙ ПАЙПЛАЙН ---
def main():
    if os.path.exists(ZIP_FILE): unpack_dataset(ZIP_FILE, OUTPUT_DIR)
    files = glob.glob(os.path.join(OUTPUT_DIR, '*.csv'))
    df_raw = pd.concat([pd.read_csv(f, sep=';' if ';' in open(f).readline() else ',').assign(symbol=os.path.basename(f)) for f in tqdm(files[:150], desc="Загрузка")], ignore_index=True)

    df_feat, all_f, base_3d = create_financial_features(filter_and_report_chunks(get_contiguous_chunks(df_raw), 100))

    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)
    split_idx = int(len(df_feat) * 0.8)
    train_df, test_df = df_feat.iloc[:split_idx], df_feat.iloc[split_idx:]

    y_test, c_test = test_df['future_return'].values, test_df['chunk_full_id'].values

    print("\n" + "="*80)
    print(f"💰 ФИНАНСОВЫЙ ОТЧЕТ (CUDA) | Старт: $10,000 | Риск: 20% | Порог: {RETURN_THRESH*100:.2f}%")
    print("="*80)

    # 1. Buy & Hold
    advanced_backtest_chunked(np.ones_like(y_test)*100, y_test, c_test, "0. Buy & Hold (Market)")

    # 2. Optimized CatBoost (With Signal)
    run_catboost(train_df, train_df['future_return'], test_df, y_test, c_test, "1. Optimized CB (Signal)", all_f)

    # 3. CatBoost (No Signal)
    f_no_rd = [f for f in all_f if 'rd_' not in f]
    run_catboost(train_df, train_df['future_return'], test_df, y_test, c_test, "2. Optimized CB (No Signal)", f_no_rd)

    # 4. Tabular DNN
    run_keras_reg(train_df, train_df['future_return'], test_df, y_test, c_test, "3. Tabular DNN (No Signal)", False, f_no_rd)

    # 5. LSTM
    run_keras_reg(train_df, train_df['future_return'], test_df, y_test, c_test, "4. LSTM (No Signal)", True, base_3d)

if __name__ == "__main__":
    main()

Лаги: 100%|██████████| 3/3 [00:00<00:00, 27.17it/s]



💰 ФИНАНСОВЫЙ ОТЧЕТ (CUDA) | Старт: $10,000 | Риск: 20% | Порог: 0.20%
[0. Buy & Hold (Market)        ] Баланс: $ 9968.07 | Профит: $  -31.93 (  -0.32%) | Сделок: 78
[1. Optimized CB (Signal)      ] Баланс: $10006.54 | Профит: $    6.54 (   0.07%) | Сделок: 63
[2. Optimized CB (No Signal)   ] Баланс: $10009.18 | Профит: $    9.18 (   0.09%) | Сделок: 23
[3. Tabular DNN (No Signal)    ] Баланс: $ 7215.75 | Профит: $-2784.25 ( -27.84%) | Сделок: 5030
[4. LSTM (No Signal)           ] Баланс: $ 9992.82 | Профит: $   -7.18 (  -0.07%) | Сделок: 6
